In [0]:
!pip install feature_engine

In [0]:
# Random-Rorest
# Regressao-logistica
# Grandient-Boosting (XGBoost, LGBoost)
# MLP (multi layer perceptron)
# SVM
# Ada-Boost
# Cat-Boost

from sklearn import tree

In [0]:
# SAMPLE

from sklearn import model_selection

df_abt = spark.table("workspace.tmw_trabalhofinal.abt_ativacao").toPandas()
df_oot = df_abt[df_abt['dtRef']==df_abt['dtRef'].max()]

df_train_test = df_abt[df_abt['dtRef']!=df_abt['dtRef'].max()]

target = 'flAtivacao'
features = df_train_test.columns.tolist()[2:-1]
X = df_train_test[features]
y = df_train_test[target]

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)


print(f"Tamanho da base Treino: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Taxa respoata: ({y_train.mean()*100:.2f}%)")

print(f"Tamanho da base Test: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Taxa resposta: ({y_test.mean()*100:.2f}%)")


In [0]:
X_train.head()

In [0]:
# EXPLORE

is_na = X_train.isna().mean()
is_na[is_na>0]

# 0 (zero):
# qtdCursosIniciados
# qtdCursosFinalizados
# github_2025
# python_2025
# sql_2025
# pandas_2025
# estatistica_2025
# qtEpisodiosConcluidos28d
# pctTendenciaFrequencia28d_56d
# pctTendenciaPontos28d_56d
# pctTendenciaPontosPositivos28d_56d
# pctTendenciaTransacoes28d_56d

# Valor máximo da base:
# DiasUltimoStreak
# avgTempoInicioUltimo
# avgTempoInicioFim
# avgDiasEntreTransacoes

In [0]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

In [0]:
# MODIFY

from feature_engine import imputation

features_0 = [
    'qtdCursosIniciados',
    'qtdCursosFinalizados',
    'github_2025',
    'python_2025',
    'sql_2025',
    'pandas_2025',
    'estatistica_2025',
    'qtEpisodiosConcluidos28d',
    'pctTendenciaFrequencia28d_56d',
    'pctTendenciaPontos28d_56d',
    'pctTendenciaPontosPositivos28d_56d',
    'pctTendenciaTransacoes28d_56d'
]

imput_0 = imputation.ArbitraryNumberImputer(
        variables=features_0, 
        arbitrary_number=0
)

imput_max_tail = imputation.EndTailImputer(
        variables=[
                'DiasUltimoStreak',
                'avgTempoInicioUltimo',
                'avgTempoInicioFim', 
                'avgDiasEntreTransacoes'],
        imputation_method='max', 
        fold=1
)

# AJUSTE DOS MÉTODOS
# imput_0.fit(X_train)
# imput_max_tail.fit(X_train)

# # APLICAÇÃO DOS MÉTODOS
# X_train_transform = imput_0.transform(X_train)
# X_train_transform = imput_max_tail.transform(X_train_transform)

# is_na = X_train_transform.isna().mean()
# is_na[is_na>0]

In [0]:
# EXPLORE

# df_train = X_train_transform.copy()
# df_train[target] = y_train
# sumario = df_train.groupby(target)[features].mean().T
# sumario['rate'] = sumario[1] / sumario[0]
# sumario

In [0]:
# MODEL
import os
import logging
os.environ["MLFLOW_DATABRICKS_TAGS_TO_SKIP"] = "extraContext"
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

import mlflow
from sklearn import ensemble
from sklearn import linear_model
from sklearn import pipeline
from sklearn import metrics

# VINCULA O SCRIPT PYTHON COM O EXPERIMENTO DO MLFLOW
mlflow.set_experiment(experiment_id=22108943160739)

#with mlflow.start_run():
#
#    mlflow.sklearn.autolog()
#
#    # clf = linear_model.LogisticRegression()
#    clf = ensemble.RandomForestClassifier(min_samples_leaf=10, n_estimators=500, random_state=42)
#    # clf = ensemble.RandomForestClassifier(min_samples_leaf=12, n_estimators=500, random_state=42)
#    # clf = ensemble.RandomForestClassifier(max_depth=3, random_state=42, n_estimators=1000)
#    # clf = ensemble.AdaBoostClassifier(n_estimators=1000, learning_rate=0.9)
#    # clf = tree.DecisionTreeClassifier(min_samples_leaf=20, random_state=42, max_depth=2)  
#
#    model_pipeline = pipeline.Pipeline(steps=[
#        ('imputacao_0', imput_0),
#        ('imput_max', imput_max_tail),
#        ('scaler', scaler),
#        ('classificador', clf) ])
#    
#    model_pipeline.fit(X_train, y_train)
#
#    y_train_pred = model_pipeline.predict(X_train)
#    y_train_prob = model_pipeline.predict_proba(X_train)[:,1]
#
#    acc_train = metrics.accuracy_score(y_train, y_train_pred)
#    auc_train = metrics.roc_auc_score(y_train, y_train_prob)
#
#    y_test_pred = model_pipeline.predict(X_test)
#    y_test_prob = model_pipeline.predict_proba(X_test)[:,1]
#
#    # y_test_pred = (y_test_prob >= y_train.mean()).astype(int)
#    acc_test = metrics.accuracy_score(y_test, y_test_pred)
#    auc_test = metrics.roc_auc_score(y_test, y_test_prob)
#
#    y_oot_pred = model_pipeline.predict(df_oot[features])
#    y_oot_prob = model_pipeline.predict_proba(df_oot[features])[:,1]
#
#    acc_oot = metrics.accuracy_score(df_oot[target], y_oot_pred)
#    auc_oot = metrics.roc_auc_score(df_oot[target], y_oot_prob)
#
#    acc_teo_cu = metrics.accuracy_score(y_test, [0 for i in range(y_test.shape[0])])
#
#    print(f"Acurácia Test: {acc_test:.4f} | Ganho: {acc_test / acc_teo_cu*100 - 100 :.2f}%")
#    print(f"AUC Test: {auc_test:.4f} | Ganho: {auc_test/0.5*100 - 100 :.2f}%")
#
#    mlflow.log_metrics({"auc_train": auc_train,
#                        "auc_test": auc_test,
#                        "auc_oot":auc_oot})
#    
#    model_pipeline.fit(X, y)
#    mlflow.sklearn.log_model(model_pipeline, name="model-refit", input_example=X.head())


In [0]:
def train_model(clf, imput_0, imput_max_tail, scaler, X_train, y_train, X_test, y_test, df_oot, X, y):

    with mlflow.start_run():

        mlflow.sklearn.autolog()

        model_pipeline = pipeline.Pipeline(steps=[
            ('imputacao_0', imput_0),
            ('imput_max', imput_max_tail),
            ('scaler', scaler),
            ('classificador', clf) ])
        
        model_pipeline.fit(X_train, y_train)

        y_train_pred = model_pipeline.predict(X_train)
        y_train_prob = model_pipeline.predict_proba(X_train)[:,1]

        acc_train = metrics.accuracy_score(y_train, y_train_pred)
        auc_train = metrics.roc_auc_score(y_train, y_train_prob)

        y_test_pred = model_pipeline.predict(X_test)
        y_test_prob = model_pipeline.predict_proba(X_test)[:,1]

        acc_test = metrics.accuracy_score(y_test, y_test_pred)
        auc_test = metrics.roc_auc_score(y_test, y_test_prob)

        y_oot_pred = model_pipeline.predict(df_oot[features])
        y_oot_prob = model_pipeline.predict_proba(df_oot[features])[:,1]

        acc_oot = metrics.accuracy_score(df_oot[target], y_oot_pred)
        auc_oot = metrics.roc_auc_score(df_oot[target], y_oot_prob)

        acc_teo_cu = metrics.accuracy_score(y_test, [0 for i in range(y_test.shape[0])])

        print(f"Acurácia Test: {acc_test:.4f} | Ganho: {acc_test / acc_teo_cu*100 - 100 :.2f}%")
        print(f"AUC Test: {auc_test:.4f} | Ganho: {auc_test/0.5*100 - 100 :.2f}%")

        mlflow.log_metrics({"auc_train": auc_train,
                            "auc_test": auc_test,
                            "auc_oot":auc_oot})
        
        model_pipeline.fit(X, y)
        mlflow.sklearn.log_model(model_pipeline, name="model-refit", input_example=X.head())


In [0]:
clf = linear_model.LogisticRegression()

train_model(clf, imput_0, imput_max_tail, scaler, X_train, y_train, X_test, y_test, df_oot, X, y)


In [0]:
import pandas as pd
pd.Series(clf.feature_importances_, index=X_train.columns.tolist()).sort_values(ascending=False)